# Aim 1: Predicting TB Non-Conversion at Month 2 / Month 6

**Goal**: Predict which TB patients at baseline will not sputum-culture convert by months 2 and 6.

**Approach**:
- Build binary classifiers using baseline clinical and demographic features
- Target: `TARGET_NON_CONVERSION_ANY` (1 = non-converter, 0 = converter)
- Models: Logistic Regression, Random Forest, XGBoost
- Evaluation: Leave-One-Out CV (due to small sample size)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from src.data.feature_engineering import prepare_aim1_data, build_preprocessor
from src.models.train import train_models, get_champion_model
from src.models.evaluate import evaluate_model, _plot_roc_curve, _plot_pr_curve

sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Load and Prepare Data

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, feature_cols = prepare_aim1_data(
    test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {feature_cols}")
print(f"\nTraining class distribution:")
print(y_train.value_counts() if len(y_train) > 0 else "No training data")

if len(X_train) > 0:
    # Show basic statistics
    print(f"\nFeature summary:")
    display(X_train.describe(include="all"))

---
## 2. Train Models with Cross-Validation

In [ ]:
if len(X_train) >= 3:
    results, trained = train_models(
        X_train, y_train, preprocessor, feature_cols,
        aim="aim1_non_conversion",
        use_smote=(y_train.sum() > 0 and y_train.sum() < len(y_train)),
        cv_folds=min(5, len(X_train)),
        random_state=42,
    )
    
    print("\nModel Performance Summary:")
    summary = pd.DataFrame([
        {
            'Model': name,
            'Version': meta['version'],
            'Train AUC': f"{meta['train_auc']:.3f}",
            'CV AUC Mean': f"{meta['cv_auc_mean']:.3f}",
            'CV AUC Std': f"{meta['cv_auc_std']:.3f}",
        }
        for name, meta in results.items()
    ])
    display(summary)
    
    # Evaluate on test set
    if len(X_test) > 0:
        print("\n=== Test Set Evaluation ===")
        for name, (pipeline, _) in trained.items():
            evaluate_model(
                pipeline, X_test, y_test, name,
                "aim1_non_conversion", results[name]['version']
            )
else:
    print("Insufficient labeled samples for training.")

---
## 3. Leave-One-Out Cross-Validation for Small Data

In [ ]:
if len(X_train) > 0 and len(X_train) <= 20:
    print(f"Running Leave-One-Out CV on {len(X_train)} samples...")
    
    loo = LeaveOneOut()
    loo_results = {}
    
    for name, (pipeline, _) in trained.items():
        y_true_loo, y_pred_loo = [], []
        for train_idx, test_idx in loo.split(X_train):
            X_loo_train = X_train.iloc[train_idx]
            y_loo_train = y_train.iloc[train_idx]
            X_loo_test = X_train.iloc[test_idx]
            y_loo_test = y_train.iloc[test_idx]
            
            p = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', pipeline.named_steps['classifier']),
            ])
            p.fit(X_loo_train, y_loo_train)
            y_pred_loo.append(p.predict(X_loo_test)[0])
            y_true_loo.append(y_loo_test.values[0])
        
        accuracy = np.mean(np.array(y_true_loo) == np.array(y_pred_loo))
        loo_results[name] = {'LOO Accuracy': accuracy}
        print(f"  {name}: LOO Accuracy = {accuracy:.3f}")
    
    print("\nLOO CV suggests model stability with small data.")

---
## 4. Feature Importance (XGBoost)

In [ ]:
if 'xgboost' in trained:
    xgb_pipeline = trained['xgboost'][0]
    
    # Get feature names after preprocessing
    try:
        preprocessor_fitted = xgb_pipeline.named_steps['preprocessor']
        xgb_model = xgb_pipeline.named_steps['classifier']
        
        # Get feature importances
        importance = xgb_model.feature_importances_
        
        fig, ax = plt.subplots(figsize=(8, 5))
        idx = np.argsort(importance)
        ax.barh(range(len(importance)), importance[idx])
        ax.set_yticks(range(len(importance)))
        ax.set_yticklabels(np.array(feature_cols)[idx])
        ax.set_xlabel('Importance')
        ax.set_title('XGBoost Feature Importance')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not extract feature importance: {e}")

---
## 5. Conclusions & Next Steps

In [ ]:
print("=== AIM 1 Summary ===")
print(f"\nData: {len(X_train)} training, {len(X_test)} test samples")
print(f"Features used: {len(feature_cols)}")
if len(X_train) >= 3:
    champion = get_champion_model('aim1_non_conversion')
    if champion:
        print(f"Champion model: {champion['model']} (v{champion['version']})")
        print(f"CV AUC: {champion['cv_auc_mean']:.3f} ± {champion['cv_auc_std']:.3f}")
    
print(f"\nKey limitations:")
print(f"1. Only {X_train is not None and len(X_train) or 0} labeled samples available")
print(f"2. Class imbalance present")
print(f"3. Results should be validated with more data")
print(f"\nRecommendations:")
print(f"1. Collect more sputum culture conversion outcomes")
print(f"2. Add FEND-TB cohort data when available")
print(f"3. Consider ensemble/staking for improved performance")